# From Co-pilot to Creator: FinSight Builder

This notebook extends the teacher-provided AI-Driven Development Lifecycle baseline for the DTS114TC Software Component. It acts as a meta-software development system: from a bounded business intent it generates SDLC documentation, UML, Flask API code, a website with an automatically generated image, tests, Docker deployment files, and CI workflow evidence.

Project boundary: the generated app uses fixed educational sample data only. It does not perform stock prediction, buy/sell/hold recommendation, trading automation, or financial advice.

## Setup & Environment

This mirrors the baseline setup stage. It checks the course-style Python environment, prepares deterministic output directories, imports the baseline helper module where available, and records optional API-key availability without printing secrets.

In [1]:
from pathlib import Path
import json
import os
import sys
import subprocess
import textwrap

# Robust project-root detection: works when the notebook is launched from Task1 or from the repository root.
if Path.cwd().name == "Task1":
    TASK1_DIR = Path.cwd()
    REPO_ROOT = TASK1_DIR.parent
else:
    REPO_ROOT = Path.cwd()
    TASK1_DIR = REPO_ROOT / "Task1"

ARTIFACTS = TASK1_DIR / "artifacts"
APP_DIR = ARTIFACTS / "app"
FLASK_DIR = APP_DIR / "flask"
TEST_DIR = APP_DIR / "tests"
DIAGRAM_DIR = ARTIFACTS / "diagrams"
STATIC_DIR = FLASK_DIR / "static"
DOCKER_DIR = APP_DIR / "docker"
CI_DIR = REPO_ROOT / ".github" / "workflows"
APP_CI_DIR = APP_DIR / ".github" / "workflows"

for path in [ARTIFACTS, APP_DIR, FLASK_DIR, TEST_DIR, DIAGRAM_DIR, STATIC_DIR, DOCKER_DIR, CI_DIR, APP_CI_DIR]:
    path.mkdir(parents=True, exist_ok=True)

try:
    import utils  # baseline helper copied from the teacher practical
    utils_status = "available"
except Exception as exc:
    utils_status = f"not imported: {exc}"

optional_keys = ["DEEPSEEK_API_KEY", "APIFREE_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"]
key_status = {name: bool(os.getenv(name)) for name in optional_keys}

print("Python:", sys.version.split()[0])
print("Repository root:", REPO_ROOT.resolve())
print("Task1 directory:", TASK1_DIR.resolve())
print("Baseline utils:", utils_status)
print("Optional API keys present:", key_status)
print("Output directories ready.")

Python: 3.10.20
Repository root: C:\Users\31075\Desktop\大二下\dts114\CW\2472811-Feiyu_Chen
Task1 directory: C:\Users\31075\Desktop\大二下\dts114\CW\2472811-Feiyu_Chen\Task1
Baseline utils: available
Optional API keys present: {'DEEPSEEK_API_KEY': False, 'APIFREE_API_KEY': False, 'OPENAI_API_KEY': False, 'GOOGLE_API_KEY': False}
Output directories ready.


# Phase 1: Inception

The baseline begins with intent and then asks AI to elaborate business problem, personas, PRD requirements, and user stories. This project keeps that pattern while making the final outputs deterministic so the marker can rerun the notebook without depending on live LLM credits.

## 1. Intent and Business Problem

In [2]:
business_problem = '''
Generate a reproducible AI-assisted software project named FinSight Builder. The notebook must automatically create SDLC documentation, UML diagrams, Flask API code, a website, a generated image, tests, Docker deployment files, and GitHub Actions CI configuration. The generated application is an educational stock watchlist and risk briefing dashboard using fixed sample data only. It must not predict prices, recommend trades, or provide financial advice.
'''.strip()

business_problem_prompt = f'''
You are supporting the DTS114TC Software Component. Expand this intent into a bounded business problem suitable for AI-DLC documentation and implementation. Preserve the constraints: no stock prediction, no buy/sell advice, no investment recommendation, and no mandatory live financial API dependency.
Intent: {business_problem}
'''.strip()

print(business_problem)
print("\nPrompt used for AI generation/human review:\n", business_problem_prompt)

Generate a reproducible AI-assisted software project named FinSight Builder. The notebook must automatically create SDLC documentation, UML diagrams, Flask API code, a website, a generated image, tests, Docker deployment files, and GitHub Actions CI configuration. The generated application is an educational stock watchlist and risk briefing dashboard using fixed sample data only. It must not predict prices, recommend trades, or provide financial advice.

Prompt used for AI generation/human review:
 You are supporting the DTS114TC Software Component. Expand this intent into a bounded business problem suitable for AI-DLC documentation and implementation. Preserve the constraints: no stock prediction, no buy/sell advice, no investment recommendation, and no mandatory live financial API dependency.
Intent: Generate a reproducible AI-assisted software project named FinSight Builder. The notebook must automatically create SDLC documentation, UML diagrams, Flask API code, a website, a generat

#### 1) AI generated - Problem statement

The following artifact represents the reviewed output of the business-problem prompt. It is written to `artifacts/problem_statement.md`.

In [3]:
problem_statement = '# Problem Statement\n\nFinSight Builder generates a Flask API and website for a stock watchlist and educational risk briefing dashboard. The system demonstrates AI-assisted SDLC work by producing requirements, UML, application code, tests, Docker deployment files, and CI workflow configuration from a controlled business problem.\n\nThe generated application uses fixed sample data and clear disclaimers. It does not predict prices, recommend trades, or provide financial advice.\n'
(ARTIFACTS / "problem_statement.md").write_text(problem_statement, encoding="utf-8")
print(problem_statement)

# Problem Statement

FinSight Builder generates a Flask API and website for a stock watchlist and educational risk briefing dashboard. The system demonstrates AI-assisted SDLC work by producing requirements, UML, application code, tests, Docker deployment files, and CI workflow configuration from a controlled business problem.

The generated application uses fixed sample data and clear disclaimers. It does not predict prices, recommend trades, or provide financial advice.



#### 2) AI generated - Personas

Personas are used as in Agile development to clarify who benefits from the generated software and what evidence each stakeholder needs.

In [4]:
personas_prompt = f'''
Using the problem statement below, create concise personas for the student developer, learner user, and coursework marker. Each persona must connect to a software requirement or assessment evidence item.
{problem_statement}
'''.strip()

personas = '# Personas\n\n## Student Developer\nNeeds a reproducible notebook showing AI-assisted documentation, design, implementation, testing, and deployment evidence.\n\n## Learner Investor\nWants a simple watchlist interface that explains educational risk factors without recommendations.\n\n## Coursework Marker\nNeeds clear evidence of SDLC structure, UML, generated image, functional API, tests, CI/CD, deployment, and version control.\n'
(ARTIFACTS / "personas.md").write_text(personas, encoding="utf-8")
print(personas_prompt)
print("\n--- personas.md ---\n")
print(personas)

Using the problem statement below, create concise personas for the student developer, learner user, and coursework marker. Each persona must connect to a software requirement or assessment evidence item.
# Problem Statement

FinSight Builder generates a Flask API and website for a stock watchlist and educational risk briefing dashboard. The system demonstrates AI-assisted SDLC work by producing requirements, UML, application code, tests, Docker deployment files, and CI workflow configuration from a controlled business problem.

The generated application uses fixed sample data and clear disclaimers. It does not predict prices, recommend trades, or provide financial advice.

--- personas.md ---

# Personas

## Student Developer
Needs a reproducible notebook showing AI-assisted documentation, design, implementation, testing, and deployment evidence.

## Learner Investor
Wants a simple watchlist interface that explains educational risk factors without recommendations.

## Coursework Marker

#### 3) AI generated - PRD and requirements

The PRD makes the project boundary explicit and maps assessment needs into implementable requirements.

In [5]:
prd_prompt = f'''
Create a product requirements document for FinSight Builder. Include scope, non-goals, functional requirements, quality requirements, and evidence requirements. Use only educational sample stock data.
Problem statement: {problem_statement}
Personas: {personas}
'''.strip()

prd = '# Product Requirements Document\n\n## Overview\nFinSight Builder is an AI-powered meta-software development notebook that generates a Flask API and website for a stock watchlist and educational risk briefing dashboard.\n\n## Goals\n- Demonstrate an AI-DLC workflow from inception through construction and operation.\n- Generate documentation, UML, Flask code, website code, image asset, tests, Docker files, and CI workflow evidence.\n- Provide educational sample stock risk summaries with an explicit non-advice disclaimer.\n\n## Non-Goals\n- Stock price prediction.\n- Buy, sell, hold, or portfolio allocation recommendations.\n- Live trading or mandatory live financial API integration.\n\n## Functional Requirements\n- Show a stock dashboard with ticker, company name, mock price, sector, risk level, and risk briefing.\n- Allow users to add supported tickers to an in-memory watchlist.\n- Provide health, stock list, stock detail, watchlist, feedback, and risk-summary API endpoints.\n- Display an automatically generated market banner image on the website.\n- Validate feedback and invalid watchlist submissions.\n\n## Non-Functional Requirements\n- Runs locally with Flask and through Docker Compose.\n- Tests can be executed with pytest.\n- CI workflow runs tests and a Docker build check.\n- All pages and API responses maintain an educational disclaimer.\n\n## Success Metrics\n- Core API tests pass locally and in CI.\n- Docker deployment serves the website and JSON endpoints on port 5005.\n- Submission package contains Task1 notebook/artifacts and Task2 evidence.\n'
(ARTIFACTS / "prd.md").write_text(prd, encoding="utf-8")
print(prd_prompt)
print("\n--- prd.md ---\n")
print(prd)

Create a product requirements document for FinSight Builder. Include scope, non-goals, functional requirements, quality requirements, and evidence requirements. Use only educational sample stock data.
Problem statement: # Problem Statement

FinSight Builder generates a Flask API and website for a stock watchlist and educational risk briefing dashboard. The system demonstrates AI-assisted SDLC work by producing requirements, UML, application code, tests, Docker deployment files, and CI workflow configuration from a controlled business problem.

The generated application uses fixed sample data and clear disclaimers. It does not predict prices, recommend trades, or provide financial advice.

Personas: # Personas

## Student Developer
Needs a reproducible notebook showing AI-assisted documentation, design, implementation, testing, and deployment evidence.

## Learner Investor
Wants a simple watchlist interface that explains educational risk factors without recommendations.

## Coursework M

#### 4) AI generated - User stories

In [6]:
user_stories_prompt = f'''
Generate user stories with acceptance criteria from this PRD. Include stories for documentation generation, UML, website/image, API use, testing, CI/CD, and Docker deployment evidence.
{prd}
'''.strip()

user_stories = [{'id': 'US-001', 'role': 'learner investor', 'story': 'I want to view a sample stock watchlist dashboard so that I can compare educational risk briefings.', 'acceptance_criteria': ['The website shows at least three stocks.', 'Each stock includes ticker, mock price, risk level, and briefing.', 'The disclaimer is visible.']}, {'id': 'US-002', 'role': 'learner investor', 'story': 'I want to add a supported ticker to my watchlist so that I can focus on examples I am studying.', 'acceptance_criteria': ['Missing ticker returns 400.', 'Known ticker returns 201.', 'Unknown ticker returns 404.']}, {'id': 'US-003', 'role': 'coursework marker', 'story': 'I want to inspect tests and deployment files so that I can verify testing, CI/CD, and Docker evidence.', 'acceptance_criteria': ['pytest tests cover core endpoints.', 'Dockerfile and docker-compose.yml exist.', 'CI workflow includes pytest and Docker build.']}]
(ARTIFACTS / "user_stories.json").write_text(json.dumps(user_stories, indent=2), encoding="utf-8")
print(user_stories_prompt)
print("\n--- user_stories.json ---")
print(json.dumps(user_stories, indent=2))

Generate user stories with acceptance criteria from this PRD. Include stories for documentation generation, UML, website/image, API use, testing, CI/CD, and Docker deployment evidence.
# Product Requirements Document

## Overview
FinSight Builder is an AI-powered meta-software development notebook that generates a Flask API and website for a stock watchlist and educational risk briefing dashboard.

## Goals
- Demonstrate an AI-DLC workflow from inception through construction and operation.
- Generate documentation, UML, Flask code, website code, image asset, tests, Docker files, and CI workflow evidence.
- Provide educational sample stock risk summaries with an explicit non-advice disclaimer.

## Non-Goals
- Stock price prediction.
- Buy, sell, hold, or portfolio allocation recommendations.
- Live trading or mandatory live financial API integration.

## Functional Requirements
- Show a stock dashboard with ticker, company name, mock price, sector, risk level, and risk briefing.
- Allow

#### 5) AI generated - API specification

In [7]:
api_spec_prompt = f'''
Create a compact API specification for the Flask application. Include endpoints, request/response purpose, validation, and the educational disclaimer.
{prd}
'''.strip()

api_spec = '# API Specification\n\nBase URL: `http://127.0.0.1:5005`\n\nEndpoints: `/health`, `/api/stocks`, `/api/stocks/<ticker>`, `/api/watchlist`, `/api/feedback`, `/api/risk-summary`, and `/`. All financial content is sample educational data only.\n'
(ARTIFACTS / "api_spec.md").write_text(api_spec, encoding="utf-8")
print(api_spec_prompt)
print("\n--- api_spec.md ---\n")
print(api_spec)

Create a compact API specification for the Flask application. Include endpoints, request/response purpose, validation, and the educational disclaimer.
# Product Requirements Document

## Overview
FinSight Builder is an AI-powered meta-software development notebook that generates a Flask API and website for a stock watchlist and educational risk briefing dashboard.

## Goals
- Demonstrate an AI-DLC workflow from inception through construction and operation.
- Generate documentation, UML, Flask code, website code, image asset, tests, Docker files, and CI workflow evidence.
- Provide educational sample stock risk summaries with an explicit non-advice disclaimer.

## Non-Goals
- Stock price prediction.
- Buy, sell, hold, or portfolio allocation recommendations.
- Live trading or mandatory live financial API integration.

## Functional Requirements
- Show a stock dashboard with ticker, company name, mock price, sector, risk level, and risk briefing.
- Allow users to add supported tickers to

# Phase 2: Construction

The baseline's Construction phase generates UML and code. This notebook writes both the design artifacts and the executable Flask project.

## 2.1 AI generated - UML diagrams

The notebook creates PlantUML sources and also creates PNG evidence. If a PlantUML renderer is available, it can be used; otherwise a deterministic local fallback image is created so the software submission is self-contained.

In [8]:
use_case_diagram = '@startuml\nleft to right direction\ntitle FinSight Builder Use Case Diagram\nactor "Student Developer" as Developer\nactor "Learner Investor" as Learner\nactor "Coursework Marker" as Marker\nrectangle "FinSight Builder" {\n  usecase "Generate SDLC docs" as UC1\n  usecase "Generate UML diagrams" as UC2\n  usecase "Generate Flask API and website" as UC3\n  usecase "View stock risk briefing" as UC4\n  usecase "Add ticker to watchlist" as UC5\n  usecase "Submit feedback" as UC6\n  usecase "Run tests and Docker deployment" as UC7\n  usecase "Review submission evidence" as UC8\n}\nDeveloper --> UC1\nDeveloper --> UC2\nDeveloper --> UC3\nDeveloper --> UC7\nLearner --> UC4\nLearner --> UC5\nLearner --> UC6\nMarker --> UC8\nMarker --> UC7\n@enduml\n'
sequence_diagram = '@startuml\ntitle FinSight Builder Stock Briefing Sequence\nactor Learner\nparticipant "Website" as Web\nparticipant "Flask API" as API\ndatabase "Sample Stock Data" as Data\nLearner -> Web: Open dashboard\nWeb -> API: GET /api/stocks\nAPI -> Data: Read sample stocks\nData --> API: Stock records\nAPI --> Web: JSON with disclaimer\nWeb --> Learner: Render cards and generated banner\nLearner -> Web: Submit ticker\nWeb -> API: POST /api/watchlist\nAPI -> Data: Validate ticker\nAPI --> Web: Watchlist update or validation error\nWeb --> Learner: Show result\n@enduml\n'
(DIAGRAM_DIR / "use_case_diagram.puml").write_text(use_case_diagram, encoding="utf-8")
(DIAGRAM_DIR / "sequence_diagram.puml").write_text(sequence_diagram, encoding="utf-8")
print("PlantUML sources written:")
print("-", DIAGRAM_DIR / "use_case_diagram.puml")
print("-", DIAGRAM_DIR / "sequence_diagram.puml")

PlantUML sources written:
- C:\Users\31075\Desktop\大二下\dts114\CW\2472811-Feiyu_Chen\Task1\artifacts\diagrams\use_case_diagram.puml
- C:\Users\31075\Desktop\大二下\dts114\CW\2472811-Feiyu_Chen\Task1\artifacts\diagrams\sequence_diagram.puml


In [9]:
from PIL import Image, ImageDraw, ImageFont

def write_diagram_png(path, title, lines):
    img = Image.new("RGB", (1200, 760), "#ffffff")
    d = ImageDraw.Draw(img)
    try:
        title_font = ImageFont.truetype("arial.ttf", 34)
        body_font = ImageFont.truetype("arial.ttf", 22)
    except OSError:
        title_font = ImageFont.load_default()
        body_font = ImageFont.load_default()
    d.rectangle((30, 30, 1170, 730), outline="#1f2937", width=3)
    d.text((60, 55), title, fill="#111827", font=title_font)
    y = 120
    for line in lines:
        d.rounded_rectangle((70, y, 1130, y + 48), radius=10, outline="#94a3b8", width=2, fill="#f8fafc")
        d.text((90, y + 12), line, fill="#111827", font=body_font)
        y += 70
    img.save(path)

write_diagram_png(
    DIAGRAM_DIR / "use_case_diagram.png",
    "FinSight Builder - Use Case Diagram",
    [
        "Student Developer -> Generate docs, UML, Flask app, tests, Docker and CI",
        "Learner Investor -> View educational risk briefing and add watchlist ticker",
        "Coursework Marker -> Review generated artifacts, tests, deployment evidence",
        "Boundary -> No prediction, trading advice, or investment recommendation",
    ],
)
write_diagram_png(
    DIAGRAM_DIR / "sequence_diagram.png",
    "FinSight Builder - Sequence Diagram",
    [
        "Learner opens website -> Website requests /api/stocks",
        "Flask API reads fixed sample stock dataset",
        "API returns JSON with educational disclaimer",
        "Website renders generated banner, cards, forms, and risk summary",
        "Learner posts watchlist/feedback -> API validates input and responds",
    ],
)
print("UML PNG evidence written.")

UML PNG evidence written.


## 2.2 AI generated - Flask API and website code

In [10]:
main_py = 'from flask import Flask, jsonify, request, send_from_directory\n\n\napp = Flask(__name__, static_folder="static")\n\nDISCLAIMER = "For educational demonstration only. Not financial advice."\n\nSTOCKS = [\n    {\n        "ticker": "AAPL",\n        "name": "Apple Inc.",\n        "sector": "Technology hardware",\n        "mock_price": 184.35,\n        "risk_level": "medium",\n        "briefing": "Revenue depends on device cycles, services growth, supply chain resilience, and consumer demand.",\n    },\n    {\n        "ticker": "MSFT",\n        "name": "Microsoft Corporation",\n        "sector": "Cloud and productivity software",\n        "mock_price": 421.16,\n        "risk_level": "low",\n        "briefing": "Diversified software and cloud revenue may reduce volatility, but AI infrastructure costs remain material.",\n    },\n    {\n        "ticker": "NVDA",\n        "name": "NVIDIA Corporation",\n        "sector": "Semiconductors",\n        "mock_price": 912.44,\n        "risk_level": "high",\n        "briefing": "AI chip demand is strong, while valuation, export controls, and supply constraints increase uncertainty.",\n    },\n    {\n        "ticker": "TSLA",\n        "name": "Tesla, Inc.",\n        "sector": "Electric vehicles",\n        "mock_price": 177.53,\n        "risk_level": "high",\n        "briefing": "Margins, delivery growth, battery costs, regulation, and leadership attention can affect sentiment.",\n    },\n]\n\nwatchlist = []\nfeedback_items = []\n\n\ndef find_stock(ticker):\n    normalized = str(ticker or "").strip().upper()\n    return next((stock for stock in STOCKS if stock["ticker"] == normalized), None)\n\n\n@app.get("/health")\ndef health():\n    return jsonify(status="ok", service="FinSight Builder", disclaimer=DISCLAIMER)\n\n\n@app.get("/api/stocks")\ndef list_stocks():\n    return jsonify(stocks=STOCKS, count=len(STOCKS), disclaimer=DISCLAIMER)\n\n\n@app.get("/api/stocks/<ticker>")\ndef stock_detail(ticker):\n    stock = find_stock(ticker)\n    if stock is None:\n        return jsonify(error="stock_not_found", disclaimer=DISCLAIMER), 404\n    return jsonify(stock=stock, disclaimer=DISCLAIMER)\n\n\n@app.post("/api/watchlist")\ndef add_watchlist_item():\n    payload = request.get_json(silent=True) or {}\n    ticker = str(payload.get("ticker", "")).strip().upper()\n    stock = find_stock(ticker)\n    if not ticker:\n        return jsonify(error="ticker is required"), 400\n    if stock is None:\n        return jsonify(error="ticker is not in the educational sample dataset"), 404\n    if ticker not in watchlist:\n        watchlist.append(ticker)\n    return jsonify(message="ticker added", watchlist=watchlist, stock=stock, disclaimer=DISCLAIMER), 201\n\n\n@app.get("/api/watchlist")\ndef get_watchlist():\n    items = [stock for stock in STOCKS if stock["ticker"] in watchlist]\n    return jsonify(watchlist=items, tickers=watchlist, disclaimer=DISCLAIMER)\n\n\n@app.post("/api/feedback")\ndef submit_feedback():\n    payload = request.get_json(silent=True) or {}\n    name = str(payload.get("name", "")).strip()\n    message = str(payload.get("message", "")).strip()\n    rating = payload.get("rating")\n    if not name or not message or rating is None:\n        return jsonify(error="name, message, and rating are required"), 400\n    try:\n        rating = int(rating)\n    except (TypeError, ValueError):\n        return jsonify(error="rating must be an integer"), 400\n    if rating < 1 or rating > 5:\n        return jsonify(error="rating must be between 1 and 5"), 400\n    item = {"id": len(feedback_items) + 1, "name": name, "message": message, "rating": rating}\n    feedback_items.append(item)\n    return jsonify(message="feedback submitted", feedback=item), 201\n\n\n@app.get("/api/feedback")\ndef list_feedback():\n    return jsonify(items=feedback_items, count=len(feedback_items))\n\n\n@app.get("/api/risk-summary")\ndef risk_summary():\n    counts = {"low": 0, "medium": 0, "high": 0}\n    for stock in STOCKS:\n        counts[stock["risk_level"]] += 1\n    return jsonify(risk_summary=counts, disclaimer=DISCLAIMER)\n\n\n@app.get("/")\ndef index():\n    return send_from_directory(".", "index.html")\n\n\n@app.get("/<path:filename>")\ndef serve_static_file(filename):\n    return send_from_directory(".", filename)\n\n\nif __name__ == "__main__":\n    app.run(host="0.0.0.0", port=5005, debug=False)\n'
index_html = '<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="UTF-8" />\n  <meta name="viewport" content="width=device-width, initial-scale=1" />\n  <title>FinSight Builder</title>\n  <style>\n    :root {\n      --bg: #f4f6f8;\n      --ink: #17202a;\n      --muted: #5c6670;\n      --line: #d8dee6;\n      --panel: #ffffff;\n      --accent: #166534;\n      --warn: #b45309;\n      --risk: #b91c1c;\n    }\n    * { box-sizing: border-box; }\n    body {\n      margin: 0;\n      font-family: Arial, Helvetica, sans-serif;\n      background: var(--bg);\n      color: var(--ink);\n      line-height: 1.5;\n    }\n    header {\n      background: #102033;\n      color: white;\n      padding: 28px 20px;\n    }\n    .wrap {\n      width: min(1120px, calc(100% - 32px));\n      margin: 0 auto;\n    }\n    h1, h2, h3 { margin: 0; }\n    h1 { font-size: clamp(2rem, 5vw, 3.5rem); }\n    h2 { font-size: 1.2rem; margin-bottom: 14px; }\n    h3 { font-size: 1rem; }\n    .subtitle {\n      color: #c8d2df;\n      max-width: 760px;\n      margin: 10px 0 0;\n    }\n    .banner {\n      width: 100%;\n      display: block;\n      aspect-ratio: 5 / 1.55;\n      object-fit: cover;\n      border-bottom: 1px solid var(--line);\n      background: #dce6ee;\n    }\n    main { padding: 24px 0 40px; }\n    .notice {\n      padding: 12px 14px;\n      border: 1px solid #f2d48a;\n      background: #fff8e6;\n      color: #5f3b00;\n      margin-bottom: 18px;\n      border-radius: 8px;\n    }\n    .grid {\n      display: grid;\n      grid-template-columns: minmax(0, 2fr) minmax(280px, 1fr);\n      gap: 18px;\n      align-items: start;\n    }\n    .panel {\n      background: var(--panel);\n      border: 1px solid var(--line);\n      border-radius: 8px;\n      padding: 18px;\n    }\n    .cards {\n      display: grid;\n      grid-template-columns: repeat(auto-fit, minmax(230px, 1fr));\n      gap: 12px;\n    }\n    .stock {\n      border: 1px solid var(--line);\n      border-radius: 8px;\n      padding: 14px;\n      background: #fbfcfd;\n    }\n    .stock-top {\n      display: flex;\n      justify-content: space-between;\n      gap: 10px;\n      align-items: start;\n    }\n    .ticker { font-size: 1.35rem; font-weight: 700; }\n    .muted { color: var(--muted); }\n    .price { font-weight: 700; margin-top: 8px; }\n    .badge {\n      display: inline-block;\n      border-radius: 999px;\n      padding: 3px 9px;\n      font-size: 0.78rem;\n      font-weight: 700;\n      text-transform: uppercase;\n      white-space: nowrap;\n    }\n    .low { background: #dcfce7; color: var(--accent); }\n    .medium { background: #fef3c7; color: var(--warn); }\n    .high { background: #fee2e2; color: var(--risk); }\n    form { display: grid; gap: 10px; }\n    label { font-weight: 700; font-size: 0.9rem; }\n    input, textarea, button {\n      width: 100%;\n      border-radius: 8px;\n      border: 1px solid var(--line);\n      padding: 10px 12px;\n      font: inherit;\n    }\n    button {\n      border: 0;\n      background: #1f6f50;\n      color: white;\n      font-weight: 700;\n      cursor: pointer;\n    }\n    button.secondary { background: #334155; }\n    .status {\n      min-height: 24px;\n      color: var(--muted);\n      font-size: 0.92rem;\n    }\n    .summary {\n      display: grid;\n      grid-template-columns: repeat(3, 1fr);\n      gap: 8px;\n      margin-bottom: 16px;\n    }\n    .summary div {\n      border: 1px solid var(--line);\n      border-radius: 8px;\n      padding: 10px;\n      text-align: center;\n      background: #fbfcfd;\n    }\n    .summary strong { display: block; font-size: 1.35rem; }\n    @media (max-width: 820px) {\n      .grid { grid-template-columns: 1fr; }\n      header { padding: 22px 16px; }\n    }\n  </style>\n</head>\n<body>\n  <header>\n    <div class="wrap">\n      <h1>FinSight Builder</h1>\n      <p class="subtitle">AI-generated Flask API and website for a sample stock watchlist and educational risk briefing dashboard.</p>\n    </div>\n  </header>\n\n  <img class="banner" src="static/generated_market_banner.png" alt="Automatically generated market dashboard banner" />\n\n  <main class="wrap">\n    <div class="notice">For educational demonstration only. Not financial advice. Prices are mock sample values.</div>\n\n    <div class="grid">\n      <section class="panel">\n        <h2>Sample Stock Briefings</h2>\n        <div id="stockCards" class="cards"></div>\n      </section>\n\n      <aside class="panel">\n        <h2>Risk Summary</h2>\n        <div class="summary">\n          <div><strong id="lowCount">0</strong><span class="muted">Low</span></div>\n          <div><strong id="mediumCount">0</strong><span class="muted">Medium</span></div>\n          <div><strong id="highCount">0</strong><span class="muted">High</span></div>\n        </div>\n\n        <h2>Add Watchlist Ticker</h2>\n        <form id="watchlistForm">\n          <label for="ticker">Ticker</label>\n          <input id="ticker" name="ticker" placeholder="AAPL" maxlength="8" required />\n          <button type="submit">Add Ticker</button>\n          <div id="watchlistStatus" class="status"></div>\n        </form>\n\n        <h2 style="margin-top:18px">Feedback</h2>\n        <form id="feedbackForm">\n          <label for="name">Name</label>\n          <input id="name" name="name" required />\n          <label for="rating">Rating 1-5</label>\n          <input id="rating" name="rating" type="number" min="1" max="5" required />\n          <label for="message">Message</label>\n          <textarea id="message" name="message" rows="3" required></textarea>\n          <button class="secondary" type="submit">Submit Feedback</button>\n          <div id="feedbackStatus" class="status"></div>\n        </form>\n      </aside>\n    </div>\n  </main>\n\n  <script>\n    const stockCards = document.getElementById("stockCards");\n    const watchlistStatus = document.getElementById("watchlistStatus");\n    const feedbackStatus = document.getElementById("feedbackStatus");\n\n    function escapeHtml(value) {\n      return String(value ?? "")\n        .replace(/&/g, "&amp;")\n        .replace(/</g, "&lt;")\n        .replace(/>/g, "&gt;")\n        .replace(/"/g, "&quot;")\n        .replace(/\'/g, "&#039;");\n    }\n\n    async function loadStocks() {\n      const [stocksResponse, riskResponse] = await Promise.all([\n        fetch("/api/stocks"),\n        fetch("/api/risk-summary")\n      ]);\n      const stocksData = await stocksResponse.json();\n      const riskData = await riskResponse.json();\n      stockCards.innerHTML = stocksData.stocks.map((stock) => `\n        <article class="stock">\n          <div class="stock-top">\n            <div>\n              <div class="ticker">${escapeHtml(stock.ticker)}</div>\n              <h3>${escapeHtml(stock.name)}</h3>\n            </div>\n            <span class="badge ${escapeHtml(stock.risk_level)}">${escapeHtml(stock.risk_level)}</span>\n          </div>\n          <div class="muted">${escapeHtml(stock.sector)}</div>\n          <div class="price">$${Number(stock.mock_price).toFixed(2)}</div>\n          <p>${escapeHtml(stock.briefing)}</p>\n        </article>\n      `).join("");\n      document.getElementById("lowCount").textContent = riskData.risk_summary.low;\n      document.getElementById("mediumCount").textContent = riskData.risk_summary.medium;\n      document.getElementById("highCount").textContent = riskData.risk_summary.high;\n    }\n\n    document.getElementById("watchlistForm").addEventListener("submit", async (event) => {\n      event.preventDefault();\n      const ticker = document.getElementById("ticker").value.trim();\n      const response = await fetch("/api/watchlist", {\n        method: "POST",\n        headers: { "Content-Type": "application/json" },\n        body: JSON.stringify({ ticker })\n      });\n      const data = await response.json();\n      watchlistStatus.textContent = response.ok\n        ? `Watchlist: ${data.watchlist.join(", ")}`\n        : data.error;\n    });\n\n    document.getElementById("feedbackForm").addEventListener("submit", async (event) => {\n      event.preventDefault();\n      const payload = {\n        name: document.getElementById("name").value.trim(),\n        rating: document.getElementById("rating").value,\n        message: document.getElementById("message").value.trim()\n      };\n      const response = await fetch("/api/feedback", {\n        method: "POST",\n        headers: { "Content-Type": "application/json" },\n        body: JSON.stringify(payload)\n      });\n      const data = await response.json();\n      feedbackStatus.textContent = response.ok ? "Feedback submitted." : data.error;\n      if (response.ok) event.target.reset();\n    });\n\n    loadStocks().catch(() => {\n      stockCards.innerHTML = "<p>Unable to load stock data. Start the Flask API and refresh.</p>";\n    });\n  </script>\n</body>\n</html>\n'
requirements_txt = 'Flask==3.0.3\n'

(FLASK_DIR / "main.py").write_text(main_py, encoding="utf-8")
(FLASK_DIR / "index.html").write_text(index_html, encoding="utf-8")
(FLASK_DIR / "requirements.txt").write_text(requirements_txt, encoding="utf-8")

print("Generated Flask app files:")
for path in [FLASK_DIR / "main.py", FLASK_DIR / "index.html", FLASK_DIR / "requirements.txt"]:
    print("-", path.relative_to(REPO_ROOT))

Generated Flask app files:
- Task1\artifacts\app\flask\main.py
- Task1\artifacts\app\flask\index.html
- Task1\artifacts\app\flask\requirements.txt


## 2.3 AI generated - Website image

This satisfies the website-with-generated-image marking item. The image is generated by code and embedded by `index.html` as `static/generated_market_banner.png`.

In [11]:
from PIL import Image, ImageDraw, ImageFont

out = STATIC_DIR / "generated_market_banner.png"
w, h = 1400, 430
img = Image.new("RGB", (w, h), "#102033")
d = ImageDraw.Draw(img)
for y in range(h):
    shade = int(32 + 55 * y / h)
    d.line((0, y, w, y), fill=(16, shade, 51))
points = [(80, 310), (210, 260), (340, 285), (470, 210), (600, 235), (730, 165), (860, 190), (990, 125), (1120, 150), (1320, 95)]
d.line(points, fill="#74c69d", width=8)
try:
    title_font = ImageFont.truetype("arial.ttf", 60)
    body_font = ImageFont.truetype("arial.ttf", 28)
except OSError:
    title_font = ImageFont.load_default()
    body_font = ImageFont.load_default()
d.text((78, 62), "FinSight Builder", fill="white", font=title_font)
d.text((82, 138), "AI-generated educational watchlist and risk briefing dashboard", fill="#c8d2df", font=body_font)
d.rounded_rectangle((980, 280, 1325, 355), radius=18, fill="#ffffff")
d.text((1008, 302), "Not financial advice", fill="#102033", font=body_font)
img.save(out)
print("Generated image:", out.relative_to(REPO_ROOT))

Generated image: Task1\artifacts\app\flask\static\generated_market_banner.png


## 2.4 AI generated - Tests

In [12]:
test_app_py = 'import sys\nfrom pathlib import Path\n\nAPP_DIR = Path(__file__).resolve().parents[1] / "flask"\nsys.path.insert(0, str(APP_DIR))\n\nfrom main import app, feedback_items, watchlist  # noqa: E402\n\n\ndef client():\n    app.config.update(TESTING=True)\n    watchlist.clear()\n    feedback_items.clear()\n    return app.test_client()\n\n\ndef test_health_endpoint():\n    response = client().get("/health")\n    assert response.status_code == 200\n    assert response.get_json()["status"] == "ok"\n\n\ndef test_stocks_have_required_fields():\n    response = client().get("/api/stocks")\n    data = response.get_json()\n    assert response.status_code == 200\n    assert data["count"] >= 3\n    for stock in data["stocks"]:\n        assert {"ticker", "name", "mock_price", "risk_level", "briefing"} <= stock.keys()\n\n\ndef test_stock_detail_and_unknown_ticker():\n    ok_response = client().get("/api/stocks/AAPL")\n    missing_response = client().get("/api/stocks/UNKNOWN")\n    assert ok_response.status_code == 200\n    assert ok_response.get_json()["stock"]["ticker"] == "AAPL"\n    assert missing_response.status_code == 404\n\n\ndef test_watchlist_validation_and_success():\n    test_client = client()\n    missing = test_client.post("/api/watchlist", json={})\n    created = test_client.post("/api/watchlist", json={"ticker": "msft"})\n    assert missing.status_code == 400\n    assert created.status_code == 201\n    assert "MSFT" in created.get_json()["watchlist"]\n\n\ndef test_feedback_validation_and_success():\n    test_client = client()\n    invalid = test_client.post("/api/feedback", json={"name": "Feiyu"})\n    valid = test_client.post(\n        "/api/feedback",\n        json={"name": "Feiyu", "message": "Clear dashboard", "rating": 5},\n    )\n    assert invalid.status_code == 400\n    assert valid.status_code == 201\n    assert valid.get_json()["feedback"]["rating"] == 5\n\n\ndef test_risk_summary_shape():\n    response = client().get("/api/risk-summary")\n    assert response.status_code == 200\n    assert set(response.get_json()["risk_summary"]) == {"low", "medium", "high"}\n'
(TEST_DIR / "test_app.py").write_text(test_app_py, encoding="utf-8")
print("Generated test file:", (TEST_DIR / "test_app.py").relative_to(REPO_ROOT))

Generated test file: Task1\artifacts\app\tests\test_app.py


# Phase 3: Operation

The baseline Operation phase prepares the system for testing, deployment, and workflow automation. This notebook writes Docker and CI files, then performs lightweight validation checks.

## 3.1 Docker and CI/CD generation

In [13]:
dockerfile = 'FROM python:3.11-slim\n\nWORKDIR /app\n\nCOPY flask/requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\n\nCOPY flask/ .\n\nEXPOSE 5005\nENV FLASK_APP=main.py\nENV FLASK_ENV=production\n\nCMD ["python", "main.py"]\n'
docker_compose = 'services:\n  finsight-builder:\n    build:\n      context: .\n      dockerfile: docker/Dockerfile\n    container_name: finsight-builder\n    ports:\n      - "5005:5005"\n    environment:\n      - FLASK_ENV=production\n      - FLASK_APP=main.py\n    restart: unless-stopped\n'
ci_yml = 'name: FinSight Builder CI\n\non:\n  push:\n  pull_request:\n\njobs:\n  test-and-build:\n    runs-on: ubuntu-latest\n    steps:\n      - name: Check out repository\n        uses: actions/checkout@v4\n\n      - name: Set up Python\n        uses: actions/setup-python@v5\n        with:\n          python-version: "3.11"\n\n      - name: Install dependencies\n        run: |\n          python -m pip install --upgrade pip\n          pip install -r Task1/artifacts/app/flask/requirements.txt pytest\n\n      - name: Run pytest\n        run: pytest Task1/artifacts/app/tests\n\n      - name: Build Docker image\n        run: docker build -f Task1/artifacts/app/docker/Dockerfile Task1/artifacts/app\n'

(DOCKER_DIR / "Dockerfile").write_text(dockerfile, encoding="utf-8")
(APP_DIR / "docker-compose.yml").write_text(docker_compose, encoding="utf-8")
(CI_DIR / "ci.yml").write_text(ci_yml, encoding="utf-8")
(APP_CI_DIR / "ci.yml").write_text(ci_yml, encoding="utf-8")

print("Docker/CI artifacts written:")
for path in [DOCKER_DIR / "Dockerfile", APP_DIR / "docker-compose.yml", CI_DIR / "ci.yml", APP_CI_DIR / "ci.yml"]:
    print("-", path.relative_to(REPO_ROOT))

Docker/CI artifacts written:
- Task1\artifacts\app\docker\Dockerfile
- Task1\artifacts\app\docker-compose.yml
- .github\workflows\ci.yml
- Task1\artifacts\app\.github\workflows\ci.yml


## 3.2 Validation commands

These commands are intentionally separated from generation so markers can run them safely in the required environment.

In [14]:
validation_commands = {
    "pytest": "python -m pytest Task1/artifacts/app/tests",
    "run_flask": "python Task1/artifacts/app/flask/main.py",
    "docker_compose": "docker compose -f Task1/artifacts/app/docker-compose.yml up --build -d",
    "website": "http://127.0.0.1:5005/",
    "health": "http://127.0.0.1:5005/health",
    "stocks": "http://127.0.0.1:5005/api/stocks",
}
print(json.dumps(validation_commands, indent=2))

{
  "pytest": "python -m pytest Task1/artifacts/app/tests",
  "run_flask": "python Task1/artifacts/app/flask/main.py",
  "docker_compose": "docker compose -f Task1/artifacts/app/docker-compose.yml up --build -d",
  "website": "http://127.0.0.1:5005/",
  "health": "http://127.0.0.1:5005/health",
  "stocks": "http://127.0.0.1:5005/api/stocks"
}


## 3.3 Software marking coverage matrix

This cell writes a coverage matrix that directly maps the Software Component marking criteria to concrete project evidence.

In [15]:
coverage_matrix = '# Software Component Coverage Matrix\n\n| Marking criterion | Evidence in this submission |\n|---|---|\n| Generate documentation | Notebook writes `problem_statement.md`, `personas.md`, `prd.md`, `user_stories.json`, and `api_spec.md`. |\n| Prompt a website with generated image | Notebook writes `index.html` and generates `static/generated_market_banner.png`; Task2 includes website screenshot. |\n| Coherent professional notebook | Notebook follows baseline AI-DLC phases: setup, Inception, Construction, Operation, validation. |\n| UML diagrams | Notebook writes PlantUML sources and PNG diagram evidence for use case and sequence views. |\n| AI-specific tooling | Prompt templates, baseline `utils.py`, optional API-key checks, deterministic fallback generation, and human-reviewed outputs are documented. |\n| Version control | Git repository uses `main` and remote `origin`; Task2 reserves commit-history screenshot evidence. |\n| Workflow | `.github/workflows/ci.yml` runs pytest and Docker image build. |\n| Testing | `Task1/artifacts/app/tests/test_app.py` covers API success and validation/error paths. |\n| Deployment | Dockerfile and Compose deploy the Flask API/website on port 5005; Task2 contains website/API/container evidence. |\n'
(ARTIFACTS / "software_rubric_coverage.md").write_text(coverage_matrix, encoding="utf-8")
print(coverage_matrix)

# Software Component Coverage Matrix

| Marking criterion | Evidence in this submission |
|---|---|
| Generate documentation | Notebook writes `problem_statement.md`, `personas.md`, `prd.md`, `user_stories.json`, and `api_spec.md`. |
| Prompt a website with generated image | Notebook writes `index.html` and generates `static/generated_market_banner.png`; Task2 includes website screenshot. |
| Coherent professional notebook | Notebook follows baseline AI-DLC phases: setup, Inception, Construction, Operation, validation. |
| UML diagrams | Notebook writes PlantUML sources and PNG diagram evidence for use case and sequence views. |
| AI-specific tooling | Prompt templates, baseline `utils.py`, optional API-key checks, deterministic fallback generation, and human-reviewed outputs are documented. |
| Version control | Git repository uses `main` and remote `origin`; Task2 reserves commit-history screenshot evidence. |
| Workflow | `.github/workflows/ci.yml` runs pytest and Docker image build

## 3.4 Final validation summary

In [16]:
required_paths = [
    TASK1_DIR / "DTS114_FinSight_Builder.ipynb",
    ARTIFACTS / "problem_statement.md",
    ARTIFACTS / "personas.md",
    ARTIFACTS / "prd.md",
    ARTIFACTS / "user_stories.json",
    ARTIFACTS / "api_spec.md",
    DIAGRAM_DIR / "use_case_diagram.puml",
    DIAGRAM_DIR / "use_case_diagram.png",
    DIAGRAM_DIR / "sequence_diagram.puml",
    DIAGRAM_DIR / "sequence_diagram.png",
    FLASK_DIR / "main.py",
    FLASK_DIR / "index.html",
    FLASK_DIR / "requirements.txt",
    STATIC_DIR / "generated_market_banner.png",
    TEST_DIR / "test_app.py",
    DOCKER_DIR / "Dockerfile",
    APP_DIR / "docker-compose.yml",
    CI_DIR / "ci.yml",
]
missing = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
notebooks = list(TASK1_DIR.rglob("*.ipynb"))
print("Missing required paths:", missing)
print("Task1 notebooks:", [p.name for p in notebooks])
print("Task1 notebook count:", len(notebooks))
assert not missing
assert len(notebooks) == 1 and notebooks[0].name == "DTS114_FinSight_Builder.ipynb"
print("Software generation validation passed.")

Missing required paths: []
Task1 notebooks: ['DTS114_FinSight_Builder.ipynb']
Task1 notebook count: 1
Software generation validation passed.
